# 🔵 ARISE Innovator — Dry Bean Starter Notebook

Welcome to the **Innovator** level of ARISE 2026!

Your job: classify images of dry beans into **7 varieties** using their
measured geometric features. This level is *deliberately harder* than Explorer
— it's about **methodology**, not just running a model.

## The rules

- You **may** use AI assistants. You must be able to explain everything.
- Use exactly `train.csv` / `test.csv` provided. Never re-split.
- Submit `submission.csv` (5/day max, best score counts).

## What "good" looks like here

- The data is **multiclass** (7 classes), **imbalanced**, and has **many
  correlated features**.
- A plain Random Forest gets ~**0.91**. To stand out you'll need feature
  selection, scaling-aware choices, hyperparameter tuning, and ensembling.
- A score of **0.93+** is good; **0.94+** is excellent.

## Your workflow

1. Load data + EDA (class balance, distributions, correlations)
2. **TODO** — feature selection (justify!)
3. **TODO** — compare at least 4 models
4. **TODO** — hyperparameter tuning
5. **TODO** — error analysis (which classes get confused?)
6. **TODO** — experiment log
7. Predict + submit


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings("ignore")
print("Setup complete!")

## 2. Load the official data

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nLabel distribution (train):")
print(train_df["label"].value_counts(normalize=True))

## 3. EDA — distributions & correlations

Run these, then **answer the questions** below in your own words.

In [ ]:
train_df.describe().T

In [ ]:
# Class balance
sns.countplot(data=train_df, x="label")
plt.xticks(rotation=45)
plt.title("Class distribution (train)")
plt.show()

In [ ]:
# Correlation between numeric features
feat_cols = [c for c in train_df.columns if c not in ("id", "label")]
corr = train_df[feat_cols].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Feature correlation heatmap")
plt.show()

# Highly correlated pairs
high = [(a, b) for a in corr.columns for b in corr.columns
        if a < b and abs(corr[a][b]) > 0.9]
print("Highly correlated pairs (>0.9):")
for a, b in high[:10]:
    print(f"  {a} <-> {b}: {corr[a][b]:.2f}")

### EDA observations (write 3+ sentences)

> TODO: What did you learn?
> 1. Which classes are rare / common (imbalance)?
> 2. Which features are strongly correlated (redundant)?
> 3. Which features look most useful for separating classes?

## 4. TODO — feature selection 🎯

The dataset has 16+ features and several are highly correlated. Decide which
features you'll keep, then **justify** your choice. You may also experiment
with selection methods (e.g. tree feature importance) — but your final
`SELECTED_FEATURES` is what the pipeline uses.

In [ ]:
# ============================================================
# YOUR DECISION — feature selection
# ============================================================
SELECTED_FEATURES = [
    # TODO: pick the features you believe are most useful
    # (start from feat_cols, then remove correlated/weak ones)
]

### Justify your feature selection (3+ sentences)

> TODO: Why drop the features you dropped? Are they redundant (correlated)?
> Do they help or hurt the model? How did you test this?

## 5. Preprocessing

Dry Bean features have **very different scales** (Area ~ 10^4, Roundness ~ 1).
Scaling matters for some models (LR, SVM, KNN) and not others (trees).
Fill in the pipeline.

In [ ]:
TARGET = "label"

X = train_df[SELECTED_FEATURES]
y = train_df[TARGET]
X_test = test_df[SELECTED_FEATURES]

# NOTE: scaling is inside the pipeline so CV stays honest
preprocessor = StandardScaler()

## 6. TODO — compare at least 4 models 🎯

Compare **at least 4** algorithms using the same 5-fold CV. Report accuracy,
**macro F1** (important for imbalanced multiclass), and training time.

In [ ]:
import time

# ============================================================
# YOUR DECISION — at least 4 models to compare
# ============================================================
MODELS = {
    # TODO: add at least 4
    # e.g. "Logistic Regression":  LogisticRegression(max_iter=2000),
    #      "Random Forest":        RandomForestClassifier(random_state=42),
    #      "Gradient Boosting":    GradientBoostingClassifier(random_state=42),
    #      "SVM (RBF)":            SVC(random_state=42),
}

results = []
for name, model in MODELS.items():
    pipe = Pipeline([("scaler", preprocessor), ("model", model)])
    t0 = time.time()
    cv_acc = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
    cv_f1 = cross_val_score(pipe, X, y, cv=5,
                            scoring="f1_macro")
    results.append({"Model": name,
                    "CV accuracy": round(cv_acc.mean(), 4),
                    "CV macro F1": round(cv_f1.mean(), 4),
                    "time (s)": round(time.time() - t0, 2)})
    print(f"{name:>22}: acc={cv_acc.mean():.4f}  f1={cv_f1.mean():.4f}")

results_df = pd.DataFrame(results).sort_values("CV macro F1", ascending=False)
results_df

### Model comparison takeaways (2+ sentences)

> TODO: Which model(s) performed best? Why? Where does scaling matter most?
> What trade-off between accuracy, F1, and training time do you see?

## 7. TODO — hyperparameter tuning 🎯

Take your best model and tune it with `GridSearchCV` (or RandomizedSearchCV).
You choose the search space — that's the point.

In [ ]:
# ============================================================
# YOUR DECISION — search space for your best model
# ============================================================
BEST_MODEL = RandomForestClassifier(random_state=42)  # TODO: your best

PARAM_GRID = {
    # TODO: define the search space
    # e.g. "n_estimators": [100, 200, 300],
    #      "max_depth": [None, 10, 20],
}

In [ ]:
search = GridSearchCV(BEST_MODEL, PARAM_GRID, cv=5,
                      scoring="f1_macro", n_jobs=-1, verbose=1)
search.fit(X, y)

print("Best params:", search.best_params_)
print("Best CV macro F1:", round(search.best_score_, 4))

### Tuning results (2+ sentences)

> TODO: What hyperparameters mattered most? Did tuning beat the untuned model?

## 8. TODO — error analysis 🎯 (this is the "great" filter)

Even good models confuse *some* classes. Fit your best model and examine the
confusion matrix. **Which two classes get confused most often, and why do you
think that is?**

In [ ]:
from sklearn.model_selection import train_test_split as tts

# Hold out 20% as a validation set for this analysis (same seed = reproducible)
X_tr, X_va, y_tr, y_va = tts(X, y, test_size=0.2, random_state=42, stratify=y)

final = Pipeline([("scaler", preprocessor), ("model", search.best_estimator_)])
final.fit(X_tr, y_tr)
y_pred_va = final.predict(X_va)

# Confusion matrix
cm = confusion_matrix(y_va, y_pred_va)
labels = sorted(y.unique())
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix (validation)")
plt.show()

print(classification_report(y_va, y_pred_va))

### Error analysis (3+ sentences)

> TODO: Which two classes does your model confuse most? Look at their features
> (size, shape) — why are they hard to tell apart? How might you reduce this?

## 9. Final model + submission

Train your tuned best model on **all** training data and predict the official test set.

In [ ]:
final.fit(X, y)
prediction = final.predict(X_test)

submission = pd.DataFrame({"id": test_df["id"], "prediction": prediction})
submission.to_csv("submission.csv", index=False)
print("submission.csv created!", submission.shape)

## 10. Self-check

In [ ]:
assert list(submission.columns) == ["id", "prediction"]
assert len(submission) == len(test_df)
assert submission["id"].equals(test_df["id"])
assert submission["prediction"].notna().all()
print("✓ Ready to upload submission.csv")

## 11. Experiment log 🧪

| # | Experiment | CV macro F1 | Result |
|---|---|---|---|
| 1 | Random Forest baseline (all features) | ~0.91 | Baseline |
| 2 | Feature selection |  |  |
| 3 | Scaling + model comparison |  |  |
| 4 | Hyperparameter tuning |  |  |
| 5 | Ensemble / ExtraTrees / etc. |  |  |

**What did you learn?** (3+ sentences)

> TODO: your honest reflection on what moved the needle and why.

---

## 🏁 The challenge

| Score | Level |
|---|---|
| 0.91 | Baseline (RF, all features) |
| 0.93+ | Good — selection + tuning |
| 0.94+ | Excellent — ensembling + deep tuning |

Your leaderboard score is 15% of the final mark; presentation + Q&A is the rest.
The experiment log is your evidence. Good luck! 🌱🔥